# CEG-WM Content V3 — Unweighted-LF operational canary handoff

This Colab notebook runs the frozen one-unit, non-roster operational canary for `content_v3_unweighted_lf_adaptive_hf_v1`. It checks out and verifies execution exact `943813ee6e667361353a9eaaf096b21a00e18398`, installs only the checked-out project's declared package, reads `CEG_WM_ROOT_KEY` and `HF_TOKEN` from Colab Secrets, and invokes the existing canary runner exactly once.

This is an engineering runtime canary only. It makes no formal scientific, calibration, fixed-FPR, or promotion claim. Create both Secrets before starting, run Cells 1–3 exactly once and in order, and do not retry or substitute another path after any failure, interruption, OOM, or runtime loss.


## 1. Fresh checkout and frozen execution identity proof

This cell must run before installation, secret access, or GPU/model work. Any existing source directory, named-branch mismatch, exact mismatch, or dirty checkout stops the handoff.


In [ ]:
import json
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
BRANCH = "stage-a-content-adaptive-dual-branch-v3-canary"
EXACT = "943813ee6e667361353a9eaaf096b21a00e18398"
RUNNER_MODULE = "experiments.run_content_adaptive_dual_branch_v3_canary"
CANARY_ID = "content-v3-unweighted-lf-full-runtime-non-roster-canary-v1"
PREFIX = "CEGWM_CANARY_RESULT"

repo = pathlib.Path("/content/cegwm-content-v3-unweighted-lf-canary-source")
HALTED = False

_ALLOWED_ERRORS = {
    "CalledProcessError", "FileExistsError", "ImportError",
    "ModuleNotFoundError", "OSError", "RuntimeError",
    "TypeError", "ValueError",
}

def stop(stage, error_class="RuntimeError"):
    global HALTED
    if HALTED:
        return
    if error_class not in _ALLOWED_ERRORS:
        error_class = "OtherOperationalError"
    line = PREFIX + " " + json.dumps(
        {
            "status": "operational_failure",
            "canary_id": CANARY_ID,
            "stage": stage,
            "error_class": error_class,
        },
        sort_keys=True, separators=(",", ":"),
    )
    HALTED = True
    print(line, flush=True)

def git(*args):
    return subprocess.run(
        ["git", *args], cwd=repo, check=True,
        capture_output=True, text=True,
    ).stdout.strip()

try:
    if repo.exists():
        raise FileExistsError
    clone = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(repo)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError
    if (
        git("branch", "--show-current") != BRANCH
        or git("rev-parse", "HEAD") != EXACT
        or git("status", "--porcelain") != ""
    ):
        raise RuntimeError
except BaseException as error:
    stop("source_checkout_identity_validation", type(error).__name__)


## 2. Install the checked-out project

This installs only the package declared by the frozen checkout, then proves that the named branch, execution exact, and clean state have not changed. Do not retry or change versions if installation fails.


In [ ]:
if not HALTED:
    try:
        install = subprocess.run(
            [sys.executable, "-m", "pip", "install", str(repo)],
            capture_output=True, text=True,
        )
        if install.returncode != 0:
            raise RuntimeError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
        ):
            raise RuntimeError
    except BaseException as error:
        stop("dependency_install_and_source_recheck", type(error).__name__)


## 3. Read Colab Secrets and invoke the operational canary once

Create Colab Secrets `CEG_WM_ROOT_KEY` and `HF_TOKEN` before running this cell. Secrets are passed only through the single child process environment and are cleared locally afterward. Raw child stdout and stderr are isolated; this cell emits only one validated, bounded `CEGWM_CANARY_RESULT` JSON line.


In [ ]:
import os

if not HALTED:
    runner_env = None
    completed = None
    runner_exception_class = None
    try:
        from google.colab import userdata

        runner_env = os.environ.copy()
        runner_env["CEG_WM_ROOT_KEY"] = userdata.get("CEG_WM_ROOT_KEY")
        runner_env["HF_TOKEN"] = userdata.get("HF_TOKEN")
        if not isinstance(runner_env["CEG_WM_ROOT_KEY"], str):
            raise RuntimeError
        if not runner_env["CEG_WM_ROOT_KEY"].strip():
            raise RuntimeError
        if not isinstance(runner_env["HF_TOKEN"], str):
            raise RuntimeError
        if not runner_env["HF_TOKEN"].strip():
            raise RuntimeError
        if (
            git("branch", "--show-current") != BRANCH
            or git("rev-parse", "HEAD") != EXACT
            or git("status", "--porcelain") != ""
        ):
            raise RuntimeError

        completed = subprocess.run(
            [
                sys.executable,
                "-m", RUNNER_MODULE,
                "--repo-root", str(repo),
                "--expected-exact", EXACT,
            ],
            cwd=repo,
            env=runner_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            text=True,
        )
    except BaseException as error:
        runner_exception_class = type(error).__name__
    finally:
        if runner_env is not None:
            runner_env.pop("CEG_WM_ROOT_KEY", None)
            runner_env.pop("HF_TOKEN", None)
        runner_env = None

    if runner_exception_class is not None or completed is None:
        stop("secret_and_runner_invocation", runner_exception_class or "RuntimeError")

if not HALTED:
    try:
        lines = completed.stdout.splitlines()
        if len(lines) != 1:
            raise RuntimeError
        line = lines[0]
        if not line.startswith(PREFIX + " ") or len(line) > 4096:
            raise RuntimeError
        payload = json.loads(line.split(" ", 1)[1])
        if not isinstance(payload, dict) or payload.get("canary_id") != CANARY_ID:
            raise RuntimeError
        if payload.get("status") not in {"operational_canary_pass", "operational_failure"}:
            raise RuntimeError
        if completed.returncode not in {0, 1}:
            raise RuntimeError
        if (completed.returncode == 0) != (payload["status"] == "operational_canary_pass"):
            raise RuntimeError
    except BaseException as error:
        stop("runner_result_validation", type(error).__name__)

if not HALTED:
    HALTED = True
    print(line, flush=True)


## Return and stop boundary

Return only the single `CEGWM_CANARY_RESULT` line. Never expose raw child stdout, stderr, traceback, Colab Secrets, or private runtime state. Do not relaunch after success, failure, interruption, OOM, dependency/model/backend error, or Colab loss. This operational canary does not produce a scientific or promotion decision.
